# P2.1 · Notebook de baselines — Cebada360
Reproducible: usa `src/` (protocolo P2.2). Ajusta `CEBADA_DATA` a la carpeta `DATASET_RETO_AGRO_2026_LIMPIO`.

In [ ]:
import os, sys, warnings; warnings.filterwarnings('ignore')
sys.path.insert(0, '..')
os.environ.setdefault('CEBADA_DATA', '../data/DATASET_RETO_AGRO_2026_LIMPIO')
import numpy as np, pandas as pd
from pathlib import Path
from src import config as C
from src.data import load, families, columns_for, EXPERIMENTS
from src.models import zoo
from src.cv import repeated_cv, spatial_cv, paired_diff
C.DATA_ROOT = Path(os.environ['CEBADA_DATA'])
tr, ev = load(); fam = families(tr); y = tr[C.TARGET].values
print(tr.shape, ev.shape); {k: len(v) for k, v in fam.items()}

## 1. Auditoría rápida del target y de la partición

In [ ]:
print(tr[C.TARGET].describe())
print(tr.groupby('estado')[C.TARGET].agg(['count','mean','std']).round(2))
print(tr[C.TARGET].round(2).value_counts().head(6))  # valores redondeados (3.0, 3.5, 2.5...) => posible captura aproximada

In [ ]:
from sklearn.neighbors import NearestNeighbors
Xc = tr[['centroid_lat','centroid_lon']].values
nn = NearestNeighbors(n_neighbors=2).fit(Xc)
d_tt = nn.kneighbors(Xc)[0][:,1]*111
d_et = nn.kneighbors(ev[['centroid_lat','centroid_lon']].values, n_neighbors=1)[0][:,0]*111
print(f'km al vecino más cercano: train→train {np.median(d_tt):.2f} | eval→train {np.median(d_et):.2f}')  # si son similares, CV aleatoria ≈ examen

## 2. Baselines por familia de features (E0–E5)
CV 5-fold × `N_REPEATS`; imputación/escalado dentro del fold.

In [ ]:
reps = 10
models = zoo(fast=True)
rows = []
for e in EXPERIMENTS:
    cols = columns_for(e, fam)
    for name, mk in models.items():
        if (e == 'E0_baseline') != (name == 'dummy_median'): continue
        X = tr[cols].values if cols else np.zeros((len(tr),1))
        rep, oof, _ = repeated_cv(mk, X, y, n_repeats=reps)
        sp,_ = spatial_cv(mk, X, y, tr['grupo_espacial'].values)
        rows.append(dict(exp=e, model=name, n_feat=len(cols), rmse=rep.rmse.mean(), rmse_sd=rep.rmse.std(), mae=rep.mae.mean(), r2=rep.r2.mean(), rmse_spatial=sp['rmse']))
lb = pd.DataFrame(rows).sort_values('rmse'); lb.round(3)

## 3. Tabla estándar de resultados (P2.2)
Columnas fijas: `exp, model, n_feat, rmse, rmse_sd, mae, r2, rmse_spatial`. Guardar en `results/leaderboard.csv`.

In [ ]:
lb.to_csv('../results/leaderboard_notebook.csv', index=False)